In [3]:
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
from scipy import sparse
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("resume-ranking")
mlflow.sklearn.autolog()

2026/09/26 10:44:15 INFO mlflow.tracking.fluent: Experiment with name 'resume-ranking' does not exist. Creating a new experiment.


In [4]:
X_train = sparse.load_npz("../data/processed/X_train.npz")
X_test = sparse.load_npz("../data/processed/X_test.npz")
y_train = np.load("../data/processed/y_train.npy")
y_test = np.load("../data/processed/y_test.npy")

df = pd.read_csv("../data/processed/resume_data_clean.csv")
fallback_ids = pd.Series("missing_resume_" + df.index.astype(str), index=df.index)
group_ids = df["skills"].fillna(fallback_ids)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=group_ids))
groups_train = group_ids.iloc[train_idx].reset_index(drop=True)

print(f"X_train: {X_train.shape} | groups_train: {groups_train.shape}")

X_train: (7519, 438) | groups_train: (7519,)


In [5]:
param_distributions = {
    "n_estimators": [200, 300, 400, 500],
    "max_depth": [10, 15, 20, None],
    "min_samples_leaf": [1, 2, 3, 5],
    "max_features": ["sqrt", "log2", 0.3],
}

gkf = GroupKFold(n_splits=5)
cv_splits = list(gkf.split(X_train, y_train, groups=groups_train))

search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=20,
    cv=cv_splits,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

In [6]:
with mlflow.start_run(run_name="random_forest_v1_tuned"):
    # Training — automatically captured by autolog (params, CV results, best model)
    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)

    final_mse = mean_squared_error(y_test, y_pred)
    final_mae = mean_absolute_error(y_test, y_pred)
    final_r2 = r2_score(y_test, y_pred)

    print("Best parameters :", search.best_params_)
    print("Best R² (CV) :", search.best_score_)
    print(f"MSE (test) : {final_mse:.4f}")
    print(f"MAE (test) : {final_mae:.4f}")
    print(f"R² (test) : {final_r2:.4f}")

    # Final test-set metrics (not captured by autolog)
    mlflow.log_metric("test_mse", final_mse)
    mlflow.log_metric("test_mae", final_mae)
    mlflow.log_metric("test_r2", final_r2)

    # Context and known limitation
    mlflow.set_tag("dataset_version", "resume_data_clean_v1")
    mlflow.set_tag("n_unique_resumes", "340")
    mlflow.set_tag(
        "known_limitation",
        "The model partly relies on domain-specific terms (e.g., autocad, civil "
        "engineering) correlated with a structural dataset bias (positions "
        "in physical engineering = lower scores), rather than solely on "
        "resume/job matching. Revisit when the dataset is expanded with "
        "more varied resumes and positions (v2)."
    )

    run_id = mlflow.active_run().info.run_id
    print(f"\nMLflow run recorded : {run_id}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits


2026/09/26 10:54:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/09/26 10:55:43 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\erb\AppData\Local\Temp\tmp4rm73l21\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.9.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/09/26 10:55:43 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/09/26 10:55:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cl

🏃 View run bouncy-cod-782 at: http://localhost:5001/#/experiments/1/runs/5cfb148aa0a94f6693cff48238949f99
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run casual-grouse-789 at: http://localhost:5001/#/experiments/1/runs/92e70d0d50604025ae942cc1e06e1952
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run entertaining-bat-503 at: http://localhost:5001/#/experiments/1/runs/5c2897f8045f4a51880265a4e4fcaf03
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run crawling-stork-387 at: http://localhost:5001/#/experiments/1/runs/b1bef940ca6248b28ae732099f013dc5
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run charming-dog-396 at: http://localhost:5001/#/experiments/1/runs/f2c283bde5254ac08f1fabedef9dc258
🧪 View experiment at: http://localhost:5001/#/experiments/1
Best parameters : {'n_estimators': 300, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'max_depth': None}
Best R² (CV) : 0.5069533243883173
MSE (test) : 0.

In [8]:
joblib.dump(best_model, "../data/processed/final_model_random_forest.pkl")
print("Final model saved for step 10 (sklearn pipeline).")

Final model saved for step 10 (sklearn pipeline).
